# Binning Bias Figure

This notebook validates the split-half cross-fold correlation estimator (`EV = 1 - mean off-diagonal cross-fold correlation`) on simulated Poisson responses to a rotated Gaussian ridge.

## Key findings (correlation estimator, 2026-09):
- The estimator recovers the ground truth at every rotation; continuous and discrete (look-up-table) tuning give indistinguishable results, so the old continuous-vs-discrete "binning bias" comparison is no longer informative and is kept below only as a legacy check.
- The only remaining bias is a **finite-sampling floor**: on an axis the neuron is invariant to, `EV` sits at `1 - reliability` (reliability = mean diagonal of the cross-fold matrix) instead of 0. The floor grows with the number of bins and shrinks with the number of samples; the reliability-normalised `EV_norm = (reliability - offdiag) / reliability` stays near 0. See the sweep at the end of the notebook (`figs/binning_noise_floor_sweep.pdf`).


## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

import os
os.chdir('/content/drive/MyDrive/ve/variance_analysis')

In [ ]:
!pip install -e /content/drive/MyDrive/ve/variance_analysis

!pip -q install fonttools

!ttx -l fonts/Helvetica.ttc

!sudo mkdir -p /usr/share/fonts/truetype/helvetica
!ttx -y 0 -o /usr/share/fonts/truetype/helvetica/Helvetica-Regular.ttf fonts/Helvetica.ttc
!ttx -y 1 -o /usr/share/fonts/truetype/helvetica/Helvetica-Oblique.ttf fonts/Helvetica.ttc
!ttx -y 2 -o /usr/share/fonts/truetype/helvetica/Helvetica-Bold.ttf fonts/Helvetica.ttc
!ttx -y 3 -o /usr/share/fonts/truetype/helvetica/Helvetica-BoldOblique.ttf fonts/Helvetica.ttc

!fc-cache -fv

In [ ]:
from pathlib import Path
import shutil
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

current_file_path = Path().resolve()
current_file_path

!mkdir -p ~/.fonts
!cp fonts/Helvetica.ttc ~/.fonts/

cache_dir = os.path.join(os.path.expanduser("~"), ".cache", "matplotlib")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'Helvetica'

In [ ]:
# Test font
fig, ax = plt.subplots(figsize=(4, 2), dpi=150)
ax.text(0.5, 0.62, 'The quick brown fox', ha='center', va='center', fontsize=16)
ax.axis('off')
plt.show()


## Imports and Core Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, zoom
from tqdm import tqdm

In [ ]:
def rotated_tuning(s1, s2, c, theta, sigma, amp, wrap=False):
    """Compute tuning response for a rotated Gaussian receptive field."""
    if wrap:
        d1 = ((s1 - c + 0.5) % 1.0) - 0.5
        d2 = ((s2 - c + 0.5) % 1.0) - 0.5
    else:
        d1 = s1 - c
        d2 = s2 - c

    ct, st = np.cos(theta), np.sin(theta)
    u = d1 * ct - d2 * st

    return amp * np.exp(-0.5 * (u / sigma) ** 2)

In [ ]:
def get_rot_counts_continuous(seed=413, theta_deg=90., N=900, num_bins=15):
    """
    Generate spike counts with CONTINUOUS tuning curve.
    The stimulus positions are discretized to bin centers, but the tuning
    curve is evaluated at these continuous positions.

    This is the ORIGINAL method from the notebook.
    """
    min_rate = 5.0
    max_rate = 35.0
    sigma = 0.1
    mu = 0.4
    lo, hi = 0.0, 1.0
    use_wrap = False

    beta0 = np.log(min_rate)
    amplitude = np.log(max_rate) - beta0
    theta = np.deg2rad(theta_deg)

    rng = np.random.default_rng(seed)
    s1 = rng.uniform(lo, hi, size=N)
    s2 = rng.uniform(lo, hi, size=N)

    # Discretize to bin centers
    bin_edges = np.linspace(lo, hi, num_bins + 1)
    bin_width = (hi - lo) / num_bins
    s1_bin = np.clip(((s1 - lo) / bin_width).astype(int), 0, num_bins - 1)
    s2_bin = np.clip(((s2 - lo) / bin_width).astype(int), 0, num_bins - 1)
    s1 = lo + (s1_bin + 0.5) * bin_width
    s2 = lo + (s2_bin + 0.5) * bin_width

    tuning = rotated_tuning(s1, s2, c=mu, theta=theta, sigma=sigma, amp=amplitude, wrap=use_wrap)
    rates = np.exp(beta0 + tuning)
    spike_counts = rng.poisson(rates)

    return spike_counts, (s1, s2), (lo, hi)


In [ ]:
def compute_discrete_rate_map(num_bins, theta_deg, mu=0.4, sigma_tc=0.1,
                               min_rate=5.0, max_rate=35.0, lo=0.0, hi=1.0,
                               integration_points=20):
    """
    Compute the expected rate for each bin by integrating the tuning curve
    over the bin's area. This gives the "true" mean rate for each bin.
    """
    beta0 = np.log(min_rate)
    amplitude = np.log(max_rate) - beta0
    theta = np.deg2rad(theta_deg)

    bin_edges = np.linspace(lo, hi, num_bins + 1)
    expected_rates = np.zeros((num_bins, num_bins))

    for i in range(num_bins):
        for j in range(num_bins):
            # Integrate over the bin using a grid
            s1_pts = np.linspace(bin_edges[i], bin_edges[i+1], integration_points)
            s2_pts = np.linspace(bin_edges[j], bin_edges[j+1], integration_points)
            s1_grid, s2_grid = np.meshgrid(s1_pts, s2_pts)

            tuning = rotated_tuning(s1_grid.ravel(), s2_grid.ravel(),
                                   c=mu, theta=theta, sigma=sigma_tc, amp=amplitude)
            rates = np.exp(beta0 + tuning)
            expected_rates[i, j] = np.mean(rates)

    return expected_rates

In [ ]:
def get_rot_counts_discrete(seed=413, theta_deg=90., N=900, num_bins=15):
    """
    Generate spike counts with DISCRETE tuning curve.
    Each bin has a fixed expected rate (the integral of the continuous tuning
    over that bin). All samples within a bin see the same expected rate.

    This should give unbiased estimates with NO smoothing since there's
    no within-bin variability in the tuning curve.
    """
    min_rate = 5.0
    max_rate = 35.0
    sigma = 0.1
    mu = 0.4
    lo, hi = 0.0, 1.0

    # Get expected rate for each bin
    expected_rates = compute_discrete_rate_map(
        num_bins, theta_deg, mu=mu, sigma_tc=sigma,
        min_rate=min_rate, max_rate=max_rate, lo=lo, hi=hi
    )

    rng = np.random.default_rng(seed)

    # Uniformly sample bin indices
    s1_bin = rng.integers(0, num_bins, size=N)
    s2_bin = rng.integers(0, num_bins, size=N)

    # Get rate for each sample based on its bin
    rates = expected_rates[s1_bin, s2_bin]
    spike_counts = rng.poisson(rates)

    # Convert bin indices to positions (bin centers)
    bin_width = (hi - lo) / num_bins
    s1 = lo + (s1_bin + 0.5) * bin_width
    s2 = lo + (s2_bin + 0.5) * bin_width

    return spike_counts, (s1, s2), (lo, hi)

In [ ]:
def build_raw_visits(s1_t, s2_t, counts, tau, num_bins, bin_edges):
    """Build raw visit arrays for cross-validation splitting."""
    visit_rates = counts.astype(float) / float(tau)

    i = np.digitize(s1_t, bin_edges) - 1
    j = np.digitize(s2_t, bin_edges) - 1
    in_range = (i >= 0) & (i < num_bins) & (j >= 0) & (j < num_bins)
    i = i[in_range]
    j = j[in_range]
    visit_rates = visit_rates[in_range]

    occ_visits, _, _ = np.histogram2d(
        s1_t[in_range], s2_t[in_range], bins=[bin_edges, bin_edges]
    )
    occ_visits = occ_visits.astype(int)
    max_bin_occ = int(occ_visits.max()) if occ_visits.size else 0

    bin_lists = [[[] for _ in range(num_bins)] for _ in range(num_bins)]
    for idx_visit, (ii, jj) in enumerate(zip(i, j)):
        bin_lists[ii][jj].append(idx_visit)

    raw_visits = np.full((num_bins, num_bins, max_bin_occ), np.nan)

    for ii in range(num_bins):
        for jj in range(num_bins):
            idxs = bin_lists[ii][jj]
            if not idxs:
                continue
            take = len(visit_rates[idxs])
            raw_visits[ii, jj, :take] = visit_rates[idxs][:take]

    return raw_visits, occ_visits

def _corr_ground_truth_from_ratemap(rate_map):
    """
    Compute correlation-based ground truth from a rate map.

    For x selectivity (axis=1): slice along y, measure how consistent
    x-tuning curves are across y values. High = strong x tuning.

    For y selectivity (axis=0): slice along x, measure how consistent
    y-tuning curves are across x values. High = strong y tuning.

    Returns (ev_x, ev_y) as normalized ratios.
    """
    def _mean_offdiag_corr(mat, axis):
        slices = np.moveaxis(mat, axis, 0)
        n = slices.shape[0]
        slices = slices.reshape(n, -1)
        # demean each slice
        slices = slices - slices.mean(axis=1, keepdims=True)
        norms = np.linalg.norm(slices, axis=1, keepdims=True)
        # treat numerically flat slices (norm ~ 0) as zero-correlation, not +/-1 noise
        flat = norms < 1e-9 * max(np.abs(mat).max(), 1e-300)
        norms = np.where(flat, 1, norms)
        slices = np.where(flat, 0.0, slices / norms)
        corr = slices @ slices.T
        off_diag = ~np.eye(n, dtype=bool)
        return np.mean(corr[off_diag])

    # EV(x) = 1 - mean off-diag corr of y-curves sliced across x (axis=0)
    ev_x = 1.0 - _mean_offdiag_corr(rate_map, axis=0)
    # EV(y) = 1 - mean off-diag corr of x-curves sliced across y (axis=1)
    ev_y = 1.0 - _mean_offdiag_corr(rate_map, axis=1)
    return float(ev_x), float(ev_y)


def ground_truth_evs_continuous(theta_deg, num_bins=15):
    """Compute ground truth for continuous case from a fine rate map."""
    rate_map = compute_discrete_rate_map(num_bins, theta_deg, integration_points=50)
    return _corr_ground_truth_from_ratemap(rate_map)


def ground_truth_evs_discrete(theta_deg, num_bins=15):
    """Compute ground truth for discrete case from the actual rate map."""
    rate_map = compute_discrete_rate_map(num_bins, theta_deg)
    return _corr_ground_truth_from_ratemap(rate_map)

## Main Experiment Functions

In [ ]:
from src.explained_variance import (
    nan_aware_gaussian_filter, masked_zoom,
    quantify_variance_corr, get_splits
)

def run_experiment(rots, n_splits, num_bins, sigma_smooth, N, use_discrete_tuning):
    """Run the full experiment with specified parameters."""
    r = len(rots)
    lo, hi = 0.0, 1.0
    bin_edges = np.linspace(lo, hi, num_bins + 1)

    # Use appropriate ground truth for each case
    if use_discrete_tuning:
        gt_ev_x = np.array([ground_truth_evs_discrete(theta, num_bins)[0] for theta in rots])
        gt_ev_y = np.array([ground_truth_evs_discrete(theta, num_bins)[1] for theta in rots])
    else:
        gt_ev_x = np.array([ground_truth_evs_continuous(theta, num_bins)[0] for theta in rots])
        gt_ev_y = np.array([ground_truth_evs_continuous(theta, num_bins)[1] for theta in rots])

    rate_maps = [None] * r
    ev_s1 = np.zeros((r, n_splits))
    ev_s2 = np.zeros((r, n_splits))

    get_counts_fn = get_rot_counts_discrete if use_discrete_tuning else get_rot_counts_continuous
    label = "discrete" if use_discrete_tuning else "continuous"

    for rot in range(r):
        for s in tqdm(range(n_splits), desc=f"θ={rots[rot]}° ({label})"):
            counts, (s1, s2), _ = get_counts_fn(
                seed=s, theta_deg=rots[rot], N=N, num_bins=num_bins
            )
            occ, _, _ = np.histogram2d(s1, s2, bins=[bin_edges, bin_edges])
            spk, _, _ = np.histogram2d(s1, s2, bins=[bin_edges, bin_edges], weights=counts)
            rate_map = np.divide(spk, occ, out=np.zeros_like(spk), where=occ > 0)
            rate_maps[rot] = rate_map

            raw_visits, _ = build_raw_visits(s1, s2, counts, 1., num_bins, bin_edges)
            _, fold0, fold1, occ_split, _ = get_splits(raw_visits[:, :, :, np.newaxis], seed=s)

            fold0_smooth = nan_aware_gaussian_filter(fold0[:, :, 0], sigma=sigma_smooth, circular_axes=(0,0))
            fold1_smooth = nan_aware_gaussian_filter(fold1[:, :, 0], sigma=sigma_smooth, circular_axes=(0,0))

            est_s1 = 1 - quantify_variance_corr(fold0_smooth, fold1_smooth, occ_split, axis=0)[0]
            est_s2 = 1 - quantify_variance_corr(fold0_smooth, fold1_smooth, occ_split, axis=1)[0]

            # raw EV per paper Eq. 2 (no normalization), each in [0,1]
            ev_s1[rot, s] = est_s1
            ev_s2[rot, s] = est_s2

    return {
        'rots': rots,
        'ev_s1': ev_s1,
        'ev_s2': ev_s2,
        'gt_ev_x': gt_ev_x,
        'gt_ev_y': gt_ev_y,
        'rate_maps': rate_maps,
        'sigma_smooth': sigma_smooth,
        'use_discrete': use_discrete_tuning,
    }

In [ ]:
def plot_results(results, title_suffix="", save_path=None):
    """Plot the rate maps and EV histograms."""
    rots = results['rots']
    ev_s1 = results['ev_s1']
    ev_s2 = results['ev_s2']
    gt_ev_x = results['gt_ev_x']
    gt_ev_y = results['gt_ev_y']
    rate_maps = results['rate_maps']

    r = len(rots)
    lo, hi = 0.0, 1.0

    fig = plt.figure(figsize=(4*r + 0.5, 7))
    gs = fig.add_gridspec(nrows=2, ncols=r + 1,
                          width_ratios=[1]*r + [0.05],
                          height_ratios=[1, 1],
                          wspace=0.25, hspace=0.35)

    vmin = min(np.nanmin(m) for m in rate_maps)
    vmax = max(np.nanmax(m) for m in rate_maps)
    im_handles = []

    for i, theta in enumerate(rots):
        ax = fig.add_subplot(gs[0, i])
        im = ax.imshow(rate_maps[i].T, origin='lower', aspect='equal',
                       extent=[lo, hi, lo, hi], cmap='viridis', vmin=vmin, vmax=vmax)
        im_handles.append(im)
        ax.set_title(f"θ={theta}°")
        if i == 0:
            ax.set_ylabel("y")
        ax.set_xlabel("x")

    cax = fig.add_subplot(gs[0, -1])
    fig.colorbar(im_handles[0], cax=cax, label='Rate')

    for i, theta in enumerate(rots):
        ax = fig.add_subplot(gs[1, i])

        h1 = ax.hist(ev_s1[i, :], bins=8, alpha=0.7, histtype='stepfilled',
                     label='x', color='C0', edgecolor='k', linewidth=0.5)
        h2 = ax.hist(ev_s2[i, :], bins=8, alpha=0.7, histtype='stepfilled',
                     label='y', color='C1', edgecolor='k', linewidth=0.5)

        ax.axvline(gt_ev_x[i], linestyle='--', linewidth=2, color='darkblue',
                   alpha=0.9, label='x GT', zorder=10)
        ax.axvline(gt_ev_y[i], linestyle='-.', linewidth=2, color='darkorange',
                   alpha=0.9, label='y GT', zorder=10)

        # Add mean markers
        mean_x = np.mean(ev_s1[i, :])
        mean_y = np.mean(ev_s2[i, :])
        ax.axvline(mean_x, linestyle=':', linewidth=1.5, color='C0', alpha=0.8)
        ax.axvline(mean_y, linestyle=':', linewidth=1.5, color='C1', alpha=0.8)

        ax.set_xlim(-0.25, 1.25)
        if i == 0:
            ax.set_ylabel("Count")
        if i == r - 1:
            ax.legend(frameon=False, loc='upper right', fontsize=8)

    plt.suptitle(f"EV Estimation {title_suffix}", fontsize=12)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')

    return fig


def print_summary(results, label=""):
    """Print summary statistics."""
    rots = results['rots']
    ev_s1 = results['ev_s1']
    ev_s2 = results['ev_s2']
    gt_ev_x = results['gt_ev_x']
    gt_ev_y = results['gt_ev_y']

    print(f"\n{'='*60}")
    print(f"Summary: {label}")
    print(f"{'='*60}")

    for i, theta in enumerate(rots):
        mean_x = np.mean(ev_s1[i, :])
        mean_y = np.mean(ev_s2[i, :])
        std_x = np.std(ev_s1[i, :])
        std_y = np.std(ev_s2[i, :])
        bias_x = mean_x - gt_ev_x[i]
        bias_y = mean_y - gt_ev_y[i]
        print(f"\nθ={theta}°:")
        print(f"  EV_x: mean={mean_x:.4f} ± {std_x:.4f} (GT={gt_ev_x[i]:.4f}, bias={bias_x:+.4f})")
        print(f"  EV_y: mean={mean_y:.4f} ± {std_y:.4f} (GT={gt_ev_y[i]:.4f}, bias={bias_y:+.4f})")

## Run Experiments

In [ ]:
# Configurationsigma
rots = [0, 25, 45]
n_splits = 600  # Increase for more stable results
num_bins = 15
N = 1000

### Continuous Tuning Curve (Original Method)

In [ ]:
# Continuous case with optimal smoothing (σ=5)
results_continuous = run_experiment(
    rots=rots, n_splits=n_splits, num_bins=num_bins,
    sigma_smooth=(0.0, 0.0), N=N, use_discrete_tuning=False
)
print_summary(results_continuous, "Continuous Tuning (σ=0.0)")


In [ ]:
fig_cont = plot_results(results_continuous, "Continuous Tuning (σ=0.0)")
plt.savefig(current_file_path / 'figs/smoothed_binning_fig_corr.pdf')
plt.show()

### Discrete Tuning Curve (legacy check)

Under the correlation estimator this is indistinguishable from the continuous case; kept for reference only.


In [ ]:
def _corr_gt(rate_map):
    def _moc(mat, axis):
        slices = np.moveaxis(mat, axis, 0)
        n = slices.shape[0]
        slices = slices.reshape(n, -1)
        slices = slices - slices.mean(axis=1, keepdims=True)
        norms = np.linalg.norm(slices, axis=1, keepdims=True)
        # treat numerically flat slices (norm ~ 0) as zero-correlation, not +/-1 noise
        flat = norms < 1e-9 * max(np.abs(mat).max(), 1e-300)
        norms = np.where(flat, 1, norms)
        slices = np.where(flat, 0.0, slices / norms)
        corr = slices @ slices.T
        off_diag = ~np.eye(n, dtype=bool)
        return np.mean(corr[off_diag])

    ev_x = 1.0 - _moc(rate_map, axis=0)
    ev_y = 1.0 - _moc(rate_map, axis=1)
    return float(ev_x), float(ev_y)

# Verify inline before anything else runs
_test_rm = compute_discrete_rate_map(15, 0)
print(f"Inline GT test θ=0°: {_corr_gt(_test_rm)}")  # should print (~1.0, ~0.0)

In [ ]:
# Discrete case with NO smoothing (σ=0)
results_discrete = run_experiment(
    rots=rots, n_splits=n_splits, num_bins=num_bins,
    sigma_smooth=(0, 0), N=N, use_discrete_tuning=True
)
print_summary(results_discrete, "Discrete Tuning (σ=0)")

In [ ]:
fig_disc = plot_results(results_discrete, "Discrete Tuning (σ=0)")
plt.savefig(current_file_path / 'figs/discrete_binning_fig.pdf')
plt.show()

## Finite-sampling floor: sweep over bin count and sample count

For a neuron tuned to `x` only (θ = 0°), the true `EV(y)` is 0. The raw estimate is `1 - offdiag`, which cannot go below `1 - reliability`: with finite data the two folds' tuning curves only partially agree even when they are identical in expectation. This sweep shows how that floor depends on the number of bins and the number of samples, and that the reliability-normalised `EV` removes it.


In [ ]:
from src.explained_variance import nan_aware_gaussian_filter, quantify_variance_corr, get_splits

def sweep_condition(num_bins, N, theta_deg=0., n_splits=100, sigma_smooth=(0.0, 0.0)):
    """Return dict of mean/sd over splits for EV_x, EV_y (raw), floor_y = 1 - reliability_y, EV_y normalised."""
    lo, hi = 0.0, 1.0
    bin_edges = np.linspace(lo, hi, num_bins + 1)
    rows = []
    for s in range(n_splits):
        counts, (s1, s2), _ = get_rot_counts_continuous(seed=s, theta_deg=theta_deg, N=N, num_bins=num_bins)
        raw_visits, _ = build_raw_visits(s1, s2, counts, 1., num_bins, bin_edges)
        _, fold0, fold1, occ_split, _ = get_splits(raw_visits[:, :, :, np.newaxis], seed=s)
        f0 = nan_aware_gaussian_filter(fold0[:, :, 0], sigma=sigma_smooth, circular_axes=(0, 0))
        f1 = nan_aware_gaussian_filter(fold1[:, :, 0], sigma=sigma_smooth, circular_axes=(0, 0))
        off_x, rel_x = quantify_variance_corr(f0, f1, occ_split, axis=0)   # slices across x -> EV(x)
        off_y, rel_y = quantify_variance_corr(f0, f1, occ_split, axis=1)   # slices across y -> EV(y)
        rows.append((1 - off_x, 1 - off_y, 1 - rel_y, (rel_y - off_y) / rel_y if rel_y > 0 else np.nan))
    rows = np.array(rows)
    return dict(ev_x=rows[:, 0], ev_y=rows[:, 1], floor_y=rows[:, 2], ev_y_norm=rows[:, 3])

BIN_SWEEP = [5, 8, 10, 15, 20, 30, 40]
N_SWEEP = [250, 500, 1000, 2000, 4000, 8000]
N_FIXED, BINS_FIXED = 1000, 15

res_bins = {b: sweep_condition(b, N_FIXED) for b in tqdm(BIN_SWEEP, desc="bin sweep (N=1000)")}
res_N = {n: sweep_condition(BINS_FIXED, n) for n in tqdm(N_SWEEP, desc="sample sweep (15 bins)")}

print(f"{'bins':>5} {'EV_x':>7} {'EV_y raw':>9} {'floor':>7} {'EV_y norm':>10}")
for b, r in res_bins.items():
    print(f"{b:>5} {np.nanmean(r['ev_x']):7.3f} {np.nanmean(r['ev_y']):9.3f} {np.nanmean(r['floor_y']):7.3f} {np.nanmean(r['ev_y_norm']):10.3f}")
print(f"\n{'N':>5} {'EV_x':>7} {'EV_y raw':>9} {'floor':>7} {'EV_y norm':>10}")
for n, r in res_N.items():
    print(f"{n:>5} {np.nanmean(r['ev_x']):7.3f} {np.nanmean(r['ev_y']):9.3f} {np.nanmean(r['floor_y']):7.3f} {np.nanmean(r['ev_y_norm']):10.3f}")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter

COL = {"ev_x": "#1f77b4", "ev_y": "#d95f02", "floor_y": "#7570b3", "ev_y_norm": "#1b9e77"}
LS = {"ev_x": "-", "ev_y": "-", "floor_y": "--", "ev_y_norm": "-"}
LAB = {"ev_x": r"$\mathrm{EV}(x)$, tuned axis", "ev_y": r"$\mathrm{EV}(y)$, untuned axis (raw)",
       "floor_y": r"floor $1-\langle C_{ii}\rangle$", "ev_y_norm": r"$\mathrm{EV}(y)$ normalised"}

def _panel(ax, xs, res, xlabel):
    for key in ["ev_x", "ev_y", "floor_y", "ev_y_norm"]:
        m = np.array([np.nanmean(res[x][key]) for x in xs])
        sd = np.array([np.nanstd(res[x][key]) for x in xs])
        ax.fill_between(xs, m - sd, m + sd, color=COL[key], alpha=0.15, linewidth=0)
        ax.plot(xs, m, color=COL[key], lw=2, ls=LS[key], marker="o", ms=4, label=LAB[key])
    ax.axhline(0, color="k", lw=0.8, alpha=0.5)
    ax.axhline(1, color="k", lw=0.8, alpha=0.5, ls=":")
    ax.set_xscale("log"); ax.set_xticks(xs); ax.set_xticklabels([str(x) for x in xs]); ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel(xlabel); ax.set_ylim(-0.1, 1.15)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="0.9", lw=0.8)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6), sharey=True)
_panel(axes[0], BIN_SWEEP, res_bins, f"bins per axis  (N = {N_FIXED} samples)")
_panel(axes[1], N_SWEEP, res_N, f"samples N  ({BINS_FIXED}$\\times${BINS_FIXED} bins)")
axes[0].set_ylabel("explained variance (θ = 0°)")
axes[0].set_title("a", loc="left", fontweight="bold"); axes[1].set_title("b", loc="left", fontweight="bold")
axes[1].legend(frameon=False, fontsize=8, loc="center right")
plt.tight_layout()
fig.savefig(current_file_path / "figs/binning_noise_floor_sweep.pdf", bbox_inches="tight")
fig.savefig(current_file_path / "figs/binning_noise_floor_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
